In [0]:
%pip install xlrd lxml html5lib --quiet

In [0]:
#import os

csv_dir = "/Workspace/Users/rimmyeb@gmail.com/크롤링/매치완료/2차_csv"
csv_files = [f"완{i})matched_2.csv" for i in range(1, 11)]
paths = [f"{csv_dir}/{fname}" for fname in csv_files]

df_list = [spark.read.format("csv")
           .option("header", True)
           .option("inferSchema", True)
           .load(path) for path in paths]

df_matched = df_list[0]
for df in df_list[1:]:
    df_matched = df_matched.unionByName(df)

print(f"합쳐진 csv 파일 수: {len(df_list)}")
display(df_matched)

In [0]:
display(df_matched.select("std_name", "lv1", "lv2", "상품명", "가격", "단위"))

In [0]:
from pyspark.sql import functions as F

# 비식재료 항목 (주방도구, 포장재, 용기, 소모품 등)
non_food_items = [
    # 주방도구/용기
    '종이컵', '소주잔', '머그', '접시', '도마', '국자', '뛝배기',
    '프라이팬', '찜솔', '트레이', '타르트팬', '미니타르트팬',
    '쿠키팬', '에어프라이어', '전자레인지', '전자렌지용',
    '락앤락', '오쿠', '테이블',
    # 포장재/소모품
    '일회용비닐봉투', '쿠킹호일', '위생봉투', '비닐', '비닐장갑',
    '위생장갑', '위생', '투명비닐', '지퍼락', '포장지',
    '기름종이', '테프론시트지', '투명테이프',
    # 도구
    '가위', '이쓰시게', '플라스틱칼', '플라스틱',
    '나무꼼이', '나무꼼치', '나무젯가락', '산적꼼지', '산적꼼치',
    '꼼치', '호박꼼지', '모양깍지', '쿠키커터',
    '실리콘', '분무기', '식힘망',
    # 기타 비식재료
    '명주실', '리봄', '참숯', '보냉가방', '밴드',
    '짤주머니', '짤주', '수건', '스폀지',
    '종이빨', '노란고무', '한지', '베보자기',
    '요리용실', '유리',
    # 색소 (식용색소 제외한 단순 색상명)
    '빨강색', '빨강', '노랑', '노랑색소', '파란색', '초록색', '주황',
    # 추가 비식재료 키워드
    '포장', '용기', '도구', '장갑', '청소', '세제', '수세미', '행주', '고무장갑',
    '스티커', '라벨', '테이프', '박스', '박스테이프', '박스라벨', '박스스티커',
    '봉투', '비닐봉투', '종이봉투', '종이', '종이박스', '플라스틱박스', '플라스틱용기',
    '알루미늄', '알루미늄박스', '알루미늄용기', '포장박스', '포장용기', '포장지', '포장재',
    '냉장고', '냉동고', '냉장', '보관', '보관용기', '보관박스', '보관봉투',
    '기구', '기기', '기계', '설비', '설치', '장치', '장비', '기구류', '기기류',
    '소모품', '소모', '소모재', '소모품류', '소모재류', '소모품박스', '소모품용기',
    '청소도구', '청소용품', '청소박스', '청소용기', '청소봉투', '청소비닐', '청소종이',
    '위생용품', '위생박스', '위생용기', '위생봉투', '위생비닐', '위생종이',
    '장식', '장식품', '장식재', '장식용품', '장식박스', '장식용기', '장식봉투',
    '장식비닐', '장식종이', '장식스티커', '장식라벨', '장식테이프',
    '색상', '색상명', '색상재', '색상용품', '색상박스', '색상용기', '색상봉투',
    '색상비닐', '색상종이', '색상스티커', '색상라벨', '색상테이프',
    '기타', '기타용품', '기타박스', '기타용기', '기타봉투', '기타비닐', '기타종이',
    '기타스티커', '기타라벨', '기타테이프',
    # 추가 검토 키워드
    '포장재료', '포장용품', '포장비닐', '포장종이', '포장스티커', '포장라벨', '포장테이프',
    '보관재', '보관용품', '보관비닐', '보관종이', '보관스티커', '보관라벨', '보관테이프',
    '청소재', '청소용품류', '청소스티커', '청소라벨', '청소테이프',
    '위생재', '위생용품류', '위생스티커', '위생라벨', '위생테이프',
    '장식재료', '장식용품류', '장식스티커', '장식라벨', '장식테이프',
    '색상재료', '색상용품류', '색상스티커', '색상라벨', '색상테이프',
    '기타재', '기타용품류', '기타스티커', '기타라벨', '기타테이프',
    '기구재', '기구용품', '기구용품류', '기구스티커', '기구라벨', '기구테이프',
    '기기재', '기기용품', '기기용품류', '기기스티커', '기기라벨', '기기테이프',
    '설비재', '설비용품', '설비용품류', '설비스티커', '설비라벨', '설비테이프',
    '장치재', '장치용품', '장치용품류', '장치스티커', '장치라벨', '장치테이프',
    '장비재', '장비용품', '장비용품류', '장비스티커', '장비라벨', '장비테이프',
    '소모재료', '소모용품', '소모용품류', '소모스티커', '소모라벨', '소모테이프',
    '없음', '불명', '미상', '미정', '미확인', '미분류', '기타분류', '기타항목'
]

# 비식재료 키워드 포함시 제거
non_food_keywords = [
    '포장', '용기', '도구', '장갑', '청소', '세제', '수세미', '행주', '고무장갑',
    '스티커', '라벨', '테이프', '박스', '봉투', '비닐', '종이', '플라스틱', '알루미늄',
    '냉장', '보관', '기구', '기기', '기계', '설비', '장치', '장비', '소모품',
    '위생', '장식', '색상', '기타', '없음', '불명', '미상', '미정', '미확인', '미분류', '기타분류', '기타항목'
]

df_filtered = df_matched.filter(
    ~F.col('std_name').isin(non_food_items) &
    ~F.col('std_name').rlike('|'.join(non_food_keywords))
)

removed_count = df_matched.count() - df_filtered.count()
print(f"원본: {df_matched.count()}행 → 필터 후: {df_filtered.count()}행 (제거: {removed_count}행)")
print(f"\n제거된 비식재료 항목:")
removed_df = df_matched.filter(
    F.col('std_name').isin(non_food_items) | F.col('std_name').rlike('|'.join(non_food_keywords))
).select('std_name').distinct()
display(removed_df)

In [0]:
display(df_matched)

In [0]:
remove_ing_ids = [
    "M00514", "M00078", "M00191", "M01321",
    "M00213", "M00253", "M00009", "M00147",
    "M00008", "M00045", "M00695", "M00164", "M00074",
    "M00443", "M00132", "M00051",
    "M00052", "M00006", "M00020",
    "M00075", "M00248", "M00120", "M00569", "M00905", "M00104",
    "M00125", "M00080", "M00645", "M00026",
    "M00343", "M00041", "M00408", "M00219", "M00092", "M00139", "M00067", "M00489",
    "M00058", "M00059", "M00024", "M00043", "M00661", "M00071",
    "M00170", "M00159", "M00042", "M00102", "M00004", "M00620", "M00285", "M00315",
    "M00030", "M00173", "M00033", "M00556",
    "M00165", "M00959", "M00038", "M00010", "M02734", "M00427",
    "M00119", "M00586", "M00456", "M00050", "M00070", "M00243", "M00031", "M00066", "M00554",
    "M00039", "M00123",
    "M00111", "M00130", "M00209",
]

# df_filtered (Cell 3에서 정의됨)에서 ing_id 기준 제거
df_filtered = df_matched.filter(~F.col("ing_id").isin(remove_ing_ids))

In [0]:
display(df_matched.select("std_name", "lv1", "lv2", "상품명", "가격", "단위"))

In [0]:
display(df_matched)

In [0]:
df_matched.printSchema()

In [0]:
from pyspark.sql.functions import col

target_cols = [
    "ing_id",
    "std_name",
    "lv1",
    "lv2",
    "freq",
    "matched_kca",
    "matched_agrofood",
    "matched_any_reference",
    "agro_category_ref",
    "kca_category_ref",
    "상품명",
    "가격",
    "단위",
    "단위_수치",
    "단위_문자",
    "score",
    "url"
]

df_matched = (
    df_matched
    .withColumn("freq", col("freq").cast("bigint"))
    .withColumn("가격", col("가격").cast("bigint"))
    .withColumn("kca_category_ref", col("kca_category_ref").cast("double"))
    .select(target_cols)
)

(
    df_matched.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze_crawling.10000recipe_crawl.ingregient1")
)

In [0]:
%skip
unique_df = df.select("std_name").distinct()
display(df_filtered)
df_filtered.write.format("delta").mode("overwrite").saveAsTable("bronze_crawling.10000recipe_crawl.ingregient1")

In [0]:
%skip
df = spark.table("bronze_crawling.10000recipe_crawl.ingregient1")
display(df.select("std_name", "lv1", "lv2", "상품명", "가격", "단위"))